In [ ]:
# ==============================================
# Cell 1: Imports
# ==============================================

from pathlib import Path
import os
import re
import json
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt, iirnotch, resample_poly

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import KFold, StratifiedKFold, StratifiedShuffleSplit

try:
    import mne
    MNE_AVAILABLE = True
except ImportError:
    MNE_AVAILABLE = False

print("TensorFlow version:", tf.__version__)
print("MNE available:", MNE_AVAILABLE)


In [ ]:
# ==============================================
# Cell 2: Reproducibility and paths
# ==============================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parents[0] if "notebooks" in str(Path.cwd()) else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CHBMIT_DIR = RAW_DIR / "chbmit"
TURKISH_DIR = RAW_DIR / "turkish"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
LOG_DIR = OUTPUT_DIR / "logs"
MODEL_DIR = OUTPUT_DIR / "models"
CACHE_DIR = OUTPUT_DIR / "cache_fold_arrays"

for d in [TABLE_DIR, FIG_DIR, LOG_DIR, MODEL_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("TABLE_DIR:", TABLE_DIR)


In [ ]:
# ==============================================
# Cell 3: Model and training configuration
# ==============================================

TRAIN_CONFIG = {
    "dataset_name": "CHB-MIT",  # change to "Turkish" when needed
    "outer_fold": 1,

    "target_fs": 256,
    "turkish_raw_fs": 500,

    "window_sec": 4.0,
    "train_stride_sec": 2.0,
    "eval_stride_sec": 4.0,
    "window_label_threshold": 0.5,

    "bandpass_low_hz": 0.5,
    "bandpass_high_hz": 70.0,
    "notch_hz": 50.0,

    "batch_size": 32,
    "epochs": 50,
    "learning_rate": 1e-4,
    "early_stopping_patience": 10,
    "reduce_lr_patience": 5,

    "mc_dropout_passes": 20,

    "threshold_grid": np.round(np.linspace(0.10, 0.90, 17), 2).tolist(),
    "fa_penalty": 0.05,

    "use_cache": True,
    "auto_regenerate_split_manifest": True,
}

MODEL_CONFIG = {
    "model_name": "FA-CNNViT",
    "random_seed": 42,

    "window_samples": 1024,
    "n_channels_chbmit": 18,  # harmonized common bipolar montage
    "n_channels_turkish": 36,

    "freq_filters": 32,
    "multi_scale_kernels": [7, 15, 31],
    "representation_dropout": 0.10,

    "cnn_filters_1": 32,
    "cnn_filters_2": 64,
    "cnn_kernel_1": (3, 7),
    "cnn_kernel_2": (3, 5),
    "cnn_dropout": 0.20,

    "patch_size": 16,
    "embed_dim": 128,
    "num_heads": 4,
    "transformer_depth": 2,
    "mlp_dim": 256,
    "transformer_dropout": 0.10,

    "fusion_dim": 128,
    "fusion_dropout": 0.25,

    "classifier_hidden": 128,
    "classifier_dropout": 0.30,

    "optimizer": "Adam",
    "loss_function": "Asymmetric Focal Loss",
}

pd.DataFrame([{"Parameter": k, "Value": str(v)} for k, v in {**TRAIN_CONFIG, **MODEL_CONFIG}.items()]).head(20)


In [ ]:
# ==============================================
# Cell 4: Save configuration tables
# ==============================================

train_cfg_path = TABLE_DIR / "fa_cnnvit_training_config.csv"
model_cfg_path = TABLE_DIR / "fa_cnnvit_model_config.csv"
train_cfg_json = LOG_DIR / "fa_cnnvit_training_config.json"
model_cfg_json = LOG_DIR / "fa_cnnvit_model_config.json"

pd.DataFrame([{"Parameter": k, "Value": str(v)} for k, v in TRAIN_CONFIG.items()]).to_csv(train_cfg_path, index=False)
pd.DataFrame([{"Parameter": k, "Value": str(v)} for k, v in MODEL_CONFIG.items()]).to_csv(model_cfg_path, index=False)

with open(train_cfg_json, "w", encoding="utf-8") as f:
    json.dump(TRAIN_CONFIG, f, indent=2)
with open(model_cfg_json, "w", encoding="utf-8") as f:
    json.dump(MODEL_CONFIG, f, indent=2)

print("Saved:")
print(" -", train_cfg_path)
print(" -", model_cfg_path)
print(" -", train_cfg_json)
print(" -", model_cfg_json)


In [ ]:
# ==============================================
# Cell 5: File indexing
# ==============================================

def is_windows_artifact(path: Path) -> bool:
    return "zone.identifier" in path.name.lower()

def list_real_files(root: Path):
    return sorted([p for p in root.rglob("*") if p.is_file() and not is_windows_artifact(p)])

chbmit_files_clean = list_real_files(CHBMIT_DIR)
turkish_files_clean = list_real_files(TURKISH_DIR)

chbmit_edf_files = sorted([p for p in chbmit_files_clean if p.suffix.lower() == ".edf"])
turkish_mat_files = sorted([p for p in turkish_files_clean if p.suffix.lower() == ".mat"])

chbmit_subject_to_edfs = defaultdict(list)
for p in chbmit_edf_files:
    chbmit_subject_to_edfs[p.parent.name.lower()].append(p)

turkish_stem_to_file = {p.stem.lower(): p for p in turkish_mat_files}

print("CHB-MIT EDF files:", len(chbmit_edf_files))
print("Indexed CHB-MIT subjects:", len(chbmit_subject_to_edfs))
print("Turkish MAT files:", len(turkish_mat_files))


In [ ]:
# ==============================================
# Cell 6: Split-manifest regeneration helper
# ==============================================

def regenerate_split_manifest_from_subject_metadata(subject_meta_df, save_path):
    split_rows = []
    N_OUTER_FOLDS = 10
    VAL_RATIO = 0.20
    RANDOM_STATE = 42

    for dataset_name_tmp in subject_meta_df["dataset"].unique():
        ds_df = subject_meta_df[subject_meta_df["dataset"] == dataset_name_tmp].reset_index(drop=True).copy()

        X = ds_df["subject_id"].values
        y_outer = ds_df["seizure_presence_subject"].values

        class_counts = pd.Series(y_outer).value_counts()
        n_unique_classes = len(class_counts)

        if n_unique_classes < 2:
            n_splits = min(N_OUTER_FOLDS, len(ds_df))
            outer_splitter = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
            outer_iterator = outer_splitter.split(X)
        else:
            min_class_count = class_counts.min()
            n_splits = min(N_OUTER_FOLDS, len(ds_df), min_class_count)
            outer_splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
            outer_iterator = outer_splitter.split(X, y_outer)

        for fold_idx, (trainval_idx, test_idx) in enumerate(outer_iterator, start=1):
            trainval_df = ds_df.iloc[trainval_idx].reset_index(drop=True)
            test_df = ds_df.iloc[test_idx].reset_index(drop=True)

            tv_X = trainval_df["subject_id"].values
            y_inner_combined = trainval_df["strata"].values if "strata" in trainval_df.columns else trainval_df["seizure_presence_subject"].values
            y_inner_presence = trainval_df["seizure_presence_subject"].values

            def safe_val_split(X_arr, y_arr, fallback_y_arr=None, val_ratio=0.20, seed=42):
                counts = pd.Series(y_arr).value_counts()
                min_count = counts.min()

                if len(counts) >= 2 and min_count >= 2:
                    splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
                    return next(splitter.split(X_arr, y_arr)), "combined_strata"

                if fallback_y_arr is not None:
                    counts_fb = pd.Series(fallback_y_arr).value_counts()
                    min_count_fb = counts_fb.min()
                    if len(counts_fb) >= 2 and min_count_fb >= 2:
                        splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
                        return next(splitter.split(X_arr, fallback_y_arr)), "seizure_presence"

                n_samples = len(X_arr)
                n_val = max(1, int(round(val_ratio * n_samples)))
                rng = np.random.default_rng(seed)
                indices = np.arange(n_samples)
                rng.shuffle(indices)
                val_idx = indices[:n_val]
                train_idx = indices[n_val:]
                return (train_idx, val_idx), "random_fallback"

            (train_idx_inner, val_idx_inner), val_strategy = safe_val_split(
                tv_X, y_inner_combined, fallback_y_arr=y_inner_presence, val_ratio=VAL_RATIO, seed=RANDOM_STATE + fold_idx
            )

            train_df = trainval_df.iloc[train_idx_inner].reset_index(drop=True)
            val_df = trainval_df.iloc[val_idx_inner].reset_index(drop=True)

            for split_name, split_df_tmp in [("train", train_df), ("val", val_df), ("test", test_df)]:
                for _, row in split_df_tmp.iterrows():
                    split_rows.append({
                        "dataset": dataset_name_tmp,
                        "outer_fold": fold_idx,
                        "split": split_name,
                        "subject_id": row["subject_id"],
                        "seizure_presence_subject": row.get("seizure_presence_subject", np.nan),
                        "seizure_burden_ratio": row.get("seizure_burden_ratio", np.nan),
                        "strata": row.get("strata", np.nan),
                        "val_split_strategy": val_strategy
                    })

    split_manifest_df = pd.DataFrame(split_rows)
    split_manifest_df.to_csv(save_path, index=False)
    return split_manifest_df


In [ ]:
# ==============================================
# Cell 7: Load or regenerate subject-wise split manifest
# ==============================================

split_manifest_path = TABLE_DIR / "split_manifest_subjectwise.csv"

if not split_manifest_path.exists():
    if TRAIN_CONFIG["auto_regenerate_split_manifest"]:
        subject_meta_path = TABLE_DIR / "subject_metadata_all.csv"
        if not subject_meta_path.exists():
            raise FileNotFoundError(
                f"{split_manifest_path} not found and {subject_meta_path} also not found.\n"
                "Please rerun 02_leakage_controlled_splits.ipynb and save subject_metadata_all.csv and split_manifest_subjectwise.csv."
            )
        subject_meta_df = pd.read_csv(subject_meta_path)
        split_manifest_df = regenerate_split_manifest_from_subject_metadata(subject_meta_df, split_manifest_path)
        print("Regenerated split manifest:", split_manifest_path)
    else:
        raise FileNotFoundError(
            f"{split_manifest_path} not found. Please save the real subject-wise split manifest first."
        )
else:
    split_manifest_df = pd.read_csv(split_manifest_path)

required_cols = {"dataset", "outer_fold", "split", "subject_id"}
missing_cols = required_cols - set(split_manifest_df.columns)
if missing_cols:
    raise ValueError(f"Loaded file is not a valid subject-wise split manifest. Missing columns: {missing_cols}")

dataset_name = TRAIN_CONFIG["dataset_name"]
outer_fold = TRAIN_CONFIG["outer_fold"]

fold_split_df = split_manifest_df[
    (split_manifest_df["dataset"] == dataset_name) &
    (split_manifest_df["outer_fold"] == outer_fold)
].copy()

if fold_split_df.empty:
    raise ValueError(f"No rows found for dataset={dataset_name}, outer_fold={outer_fold}")

print("Loaded split manifest:", split_manifest_path)
print(split_manifest_df.head())
print("\nSelected dataset:", dataset_name)
print("Selected fold:", outer_fold)
print(fold_split_df["split"].value_counts())


In [ ]:
# ==============================================
# Cell 8: Common CHB-MIT channel set and diagnostics
# ==============================================

CHBMIT_COMMON_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ"
]

print("Using harmonized CHB-MIT channel set:")
print(CHBMIT_COMMON_CHANNELS)
print("Count:", len(CHBMIT_COMMON_CHANNELS))

# quick diagnostic of raw channel counts
channel_count_map = defaultdict(int)
for edf_path in chbmit_edf_files:
    raw_tmp = mne.io.read_raw_edf(edf_path, preload=False, verbose="ERROR")
    channel_count_map[len(raw_tmp.ch_names)] += 1

print("Raw CHB-MIT channel-count distribution:")
for n_ch, count in sorted(channel_count_map.items()):
    print(f" - {n_ch} channels: {count} files")


In [ ]:
# ==============================================
# Cell 9: Robust CHB-MIT harmonized loader
# ==============================================

EXCLUDED_CHBMIT_FILES = []

def standardize_chbmit_channel_name(ch_name: str) -> str:
    ch = ch_name.strip().upper()
    ch = ch.replace("EEG ", "")
    ch = ch.replace("EKG ", "")
    ch = ch.replace("ECG ", "")
    ch = ch.replace("-REF", "")
    ch = ch.replace("-LE", "")
    ch = re.sub(r"-(\d+)$", "", ch)
    ch = re.sub(r"\s+", " ", ch).strip()
    return ch

def load_chbmit_harmonized(edf_path, canonical_channels, allow_exclusion=True):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose="ERROR")

    available_map = {}
    for raw_name in raw.ch_names:
        std_name = standardize_chbmit_channel_name(raw_name)
        if std_name not in available_map:
            available_map[std_name] = raw_name

    missing = [ch for ch in canonical_channels if ch.upper() not in available_map]

    if missing:
        msg = {
            "file": str(edf_path),
            "missing_channels": missing,
            "available_channels": raw.ch_names
        }
        if allow_exclusion:
            EXCLUDED_CHBMIT_FILES.append(msg)
            return None, None, None
        raise ValueError(f"Missing harmonized channels in {edf_path.name}: {missing}")

    selected_raw_names = [available_map[ch.upper()] for ch in canonical_channels]
    raw.pick(selected_raw_names)

    x = raw.get_data().astype(np.float32)
    fs = float(raw.info["sfreq"])
    return x, fs, selected_raw_names


In [ ]:
# ==============================================
# Cell 10: Deterministic preprocessing helpers
# ==============================================

def bandpass_filter(x, fs, low=0.5, high=70.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, x, axis=-1)

def notch_filter(x, fs, freq=50.0, q=30.0):
    nyq = 0.5 * fs
    if freq >= nyq:
        return x
    b, a = iirnotch(w0=freq / nyq, Q=q)
    return filtfilt(b, a, x, axis=-1)

def resample_signal(x, orig_fs, target_fs):
    if float(orig_fs) == float(target_fs):
        return x
    return resample_poly(x, up=int(target_fs), down=int(orig_fs), axis=-1)

def preprocess_recording(x, fs, cfg):
    x = notch_filter(x, fs=fs, freq=cfg["notch_hz"])
    x = bandpass_filter(x, fs=fs, low=cfg["bandpass_low_hz"], high=cfg["bandpass_high_hz"])
    x = resample_signal(x, orig_fs=fs, target_fs=cfg["target_fs"])
    return x.astype(np.float32)

def fit_channelwise_zscore_from_sum(sum_x, sumsq_x, count_x):
    mean = sum_x / count_x
    var = (sumsq_x / count_x) - (mean ** 2)
    var = np.maximum(var, 1e-8)
    std = np.sqrt(var)
    return mean[:, None], std[:, None]

def apply_channelwise_zscore(x, mean, std):
    return (x - mean) / std

def generate_windows(x, fs, window_sec=4.0, stride_sec=2.0):
    window_size = int(window_sec * fs)
    stride_size = int(stride_sec * fs)

    windows = []
    starts = []

    for start in range(0, x.shape[1] - window_size + 1, stride_size):
        end = start + window_size
        windows.append(x[:, start:end])
        starts.append(start)

    if len(windows) == 0:
        return np.empty((0, x.shape[0], window_size), dtype=np.float32), []

    return np.stack(windows).astype(np.float32), starts

def label_window_from_mask(mask_segment, threshold=0.5):
    frac = np.mean(mask_segment)
    return int(frac >= threshold)


In [ ]:
# ==============================================
# Cell 11: CHB-MIT seizure interval parsing
# ==============================================

def parse_chbmit_summary(summary_path):
    summary_path = Path(summary_path)
    if not summary_path.exists():
        return {}

    text = summary_path.read_text(encoding="utf-8", errors="ignore")
    block_pattern = re.compile(
        r"File Name:\s*(?P<file>[^\n]+)\n(?P<body>.*?)(?=(?:\nFile Name:)|\Z)",
        re.DOTALL
    )

    out = {}
    for match in block_pattern.finditer(text):
        file_name = match.group("file").strip()
        body = match.group("body")
        recording_stem = Path(file_name).stem.lower()

        starts = re.findall(r"Seizure\s+\d+\s+Start Time:\s*(\d+)\s*seconds", body)
        ends = re.findall(r"Seizure\s+\d+\s+End Time:\s*(\d+)\s*seconds", body)

        intervals = [(int(s), int(e)) for s, e in zip(starts, ends)]
        out[recording_stem] = intervals
    return out

chbmit_summary_cache = {}
for subject, edfs in chbmit_subject_to_edfs.items():
    summary_path = CHBMIT_DIR / subject / f"{subject}-summary.txt"
    chbmit_summary_cache[subject] = parse_chbmit_summary(summary_path)

print("Parsed CHB-MIT summaries for", len(chbmit_summary_cache), "subjects")


In [ ]:
# ==============================================
# Cell 12: Turkish reconstruction helpers
# ==============================================

def reconstruct_turkish_trials(mat_path):
    mat_dict = sio.loadmat(mat_path)
    data_obj = mat_dict["data"]
    labels = mat_dict.get("label", None)

    n_trials, n_channels = data_obj.shape
    trial_list = []

    for trial_idx in range(n_trials):
        channel_signals = []
        for ch_idx in range(n_channels):
            sig = data_obj[trial_idx, ch_idx]
            sig = np.asarray(sig).squeeze().astype(np.float32)
            if sig.ndim != 1:
                raise ValueError(f"Unexpected Turkish signal shape at trial={trial_idx}, ch={ch_idx}: {sig.shape}")
            channel_signals.append(sig)

        trial_arr = np.stack(channel_signals, axis=0)  # (channels, time)
        trial_list.append(trial_arr)

    trials = np.stack(trial_list, axis=0)  # (n_trials, n_channels, n_time)

    if labels is not None:
        labels = np.asarray(labels).squeeze().astype(np.int32)

    return trials, labels


In [ ]:
# ==============================================
# Cell 13: Fit train-only normalization statistics
# ==============================================

def fit_normalization_stats_for_fold(dataset_name, fold_split_df, cfg):
    train_subjects = fold_split_df[fold_split_df["split"] == "train"]["subject_id"].tolist()

    sum_x = None
    sumsq_x = None
    count_x = 0

    if dataset_name == "CHB-MIT":
        for subject_id in train_subjects:
            for edf_path in chbmit_subject_to_edfs.get(subject_id, []):
                x, fs_raw, _ = load_chbmit_harmonized(
                    edf_path,
                    canonical_channels=CHBMIT_COMMON_CHANNELS,
                    allow_exclusion=True
                )
                if x is None:
                    continue

                x = preprocess_recording(x, fs=fs_raw, cfg=cfg)

                if sum_x is None:
                    n_channels = x.shape[0]
                    sum_x = np.zeros(n_channels, dtype=np.float64)
                    sumsq_x = np.zeros(n_channels, dtype=np.float64)

                if x.shape[0] != sum_x.shape[0]:
                    raise ValueError(
                        f"Channel mismatch after harmonization in {edf_path.name}: "
                        f"{x.shape[0]} vs expected {sum_x.shape[0]}"
                    )

                sum_x += np.sum(x, axis=1)
                sumsq_x += np.sum(x ** 2, axis=1)
                count_x += x.shape[1]

    elif dataset_name == "Turkish":
        for subject_id in train_subjects:
            mat_path = turkish_stem_to_file[subject_id.lower()]
            trials, labels = reconstruct_turkish_trials(mat_path)

            for trial_idx in range(trials.shape[0]):
                x = preprocess_recording(trials[trial_idx].astype(np.float32), fs=cfg["turkish_raw_fs"], cfg=cfg)

                if sum_x is None:
                    n_channels = x.shape[0]
                    sum_x = np.zeros(n_channels, dtype=np.float64)
                    sumsq_x = np.zeros(n_channels, dtype=np.float64)

                if x.shape[0] != sum_x.shape[0]:
                    raise ValueError(
                        f"Turkish channel mismatch at subject={subject_id}, trial={trial_idx}: "
                        f"{x.shape[0]} vs expected {sum_x.shape[0]}"
                    )

                sum_x += np.sum(x, axis=1)
                sumsq_x += np.sum(x ** 2, axis=1)
                count_x += x.shape[1]
    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")

    if count_x == 0:
        raise RuntimeError("No valid signals were available for normalization fitting.")

    mean, std = fit_channelwise_zscore_from_sum(sum_x, sumsq_x, count_x)
    return mean.astype(np.float32), std.astype(np.float32)

norm_mean, norm_std = fit_normalization_stats_for_fold(dataset_name, fold_split_df, TRAIN_CONFIG)
print("Normalization mean shape:", norm_mean.shape)
print("Normalization std shape:", norm_std.shape)

if len(EXCLUDED_CHBMIT_FILES) > 0:
    excluded_df = pd.DataFrame(EXCLUDED_CHBMIT_FILES)
    excluded_path = TABLE_DIR / f"excluded_chbmit_files_fold{TRAIN_CONFIG['outer_fold']}.csv"
    excluded_df.to_csv(excluded_path, index=False)
    print(f"Excluded {len(EXCLUDED_CHBMIT_FILES)} incompatible CHB-MIT files.")
    print("Saved exclusion log:", excluded_path)
    display(excluded_df.head())
else:
    print("No CHB-MIT files were excluded during normalization fitting.")


In [ ]:
# ==============================================
# Cell 14: Build train/val/test arrays for one fold
# ==============================================

def build_arrays_for_fold(dataset_name, fold_split_df, cfg, norm_mean, norm_std):
    cache_path = CACHE_DIR / f"{dataset_name.lower().replace('-', '_')}_fold{cfg['outer_fold']}.npz"

    if cfg["use_cache"] and cache_path.exists():
        data = np.load(cache_path, allow_pickle=True)
        print("Loaded cached arrays:", cache_path)
        return {
            "X_train": data["X_train"],
            "y_train": data["y_train"],
            "X_val": data["X_val"],
            "y_val": data["y_val"],
            "X_test": data["X_test"],
            "y_test": data["y_test"],
        }

    split_to_subjects = {
        split_name: fold_split_df[fold_split_df["split"] == split_name]["subject_id"].tolist()
        for split_name in ["train", "val", "test"]
    }

    out = {}

    for split_name in ["train", "val", "test"]:
        X_list = []
        y_list = []

        stride_sec = cfg["train_stride_sec"] if split_name == "train" else cfg["eval_stride_sec"]

        if dataset_name == "CHB-MIT":
            for subject_id in split_to_subjects[split_name]:
                summary_map = chbmit_summary_cache.get(subject_id, {})

                for edf_path in chbmit_subject_to_edfs[subject_id]:
                    x, fs_raw, _ = load_chbmit_harmonized(
                        edf_path,
                        canonical_channels=CHBMIT_COMMON_CHANNELS,
                        allow_exclusion=True
                    )
                    if x is None:
                        continue

                    x = preprocess_recording(x, fs=fs_raw, cfg=cfg)
                    x = apply_channelwise_zscore(x, norm_mean, norm_std)

                    recording_stem = edf_path.stem.lower()
                    intervals = summary_map.get(recording_stem, [])

                    seizure_mask = np.zeros(x.shape[1], dtype=np.float32)
                    for start_sec, end_sec in intervals:
                        s = int(start_sec * cfg["target_fs"])
                        e = int(end_sec * cfg["target_fs"])
                        s = max(0, min(s, x.shape[1]))
                        e = max(0, min(e, x.shape[1]))
                        seizure_mask[s:e] = 1.0

                    windows, starts = generate_windows(
                        x,
                        fs=cfg["target_fs"],
                        window_sec=cfg["window_sec"],
                        stride_sec=stride_sec
                    )

                    for i, start in enumerate(starts):
                        end = start + int(cfg["window_sec"] * cfg["target_fs"])
                        y_i = label_window_from_mask(seizure_mask[start:end], threshold=cfg["window_label_threshold"])
                        X_list.append(windows[i])
                        y_list.append(y_i)

        elif dataset_name == "Turkish":
            for subject_id in split_to_subjects[split_name]:
                mat_path = turkish_stem_to_file[subject_id.lower()]
                trials, labels = reconstruct_turkish_trials(mat_path)

                for trial_idx in range(trials.shape[0]):
                    x = preprocess_recording(trials[trial_idx].astype(np.float32), fs=cfg["turkish_raw_fs"], cfg=cfg)
                    x = apply_channelwise_zscore(x, norm_mean, norm_std)

                    y_trial = int(labels[trial_idx]) if labels is not None else int("epile" in mat_path.parent.name.lower())

                    windows, starts = generate_windows(
                        x,
                        fs=cfg["target_fs"],
                        window_sec=cfg["window_sec"],
                        stride_sec=stride_sec
                    )

                    for i in range(windows.shape[0]):
                        X_list.append(windows[i])
                        y_list.append(y_trial)
        else:
            raise ValueError(f"Unsupported dataset: {dataset_name}")

        if len(X_list) == 0:
            raise RuntimeError(f"No windows generated for split={split_name} in dataset={dataset_name}, fold={cfg['outer_fold']}")

        X = np.stack(X_list).astype(np.float32)
        y = np.asarray(y_list).astype(np.float32)
        X = X[..., np.newaxis]

        out[f"X_{split_name}"] = X
        out[f"y_{split_name}"] = y

        print(f"{split_name}: X shape = {X.shape}, y shape = {y.shape}, positive ratio = {y.mean():.4f}")

    if cfg["use_cache"]:
        np.savez_compressed(cache_path, **out)
        print("Saved cached arrays:", cache_path)

    return out

fold_arrays = build_arrays_for_fold(dataset_name, fold_split_df, TRAIN_CONFIG, norm_mean, norm_std)


In [ ]:
# ==============================================
# Cell 15: FA-CNNViT model blocks (self-contained)
# ==============================================

def conv_bn_act(x, filters, kernel_size, strides=(1, 1), dropout_rate=0.0, name_prefix="conv"):
    x = layers.Conv2D(filters=filters, kernel_size=kernel_size, strides=strides,
                      padding="same", use_bias=False, name=f"{name_prefix}_conv")(x)
    x = layers.BatchNormalization(name=f"{name_prefix}_bn")(x)
    x = layers.Activation("relu", name=f"{name_prefix}_relu")(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop")(x)
    return x

class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout_rate
        )
        self.drop1 = layers.Dropout(dropout_rate)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp1 = layers.Dense(mlp_dim, activation="gelu")
        self.drop2 = layers.Dropout(dropout_rate)
        self.mlp2 = layers.Dense(embed_dim)
        self.drop3 = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        x1 = self.norm1(x)
        attn_out = self.attn(x1, x1, training=training)
        attn_out = self.drop1(attn_out, training=training)
        x = x + attn_out

        x2 = self.norm2(x)
        mlp_out = self.mlp1(x2)
        mlp_out = self.drop2(mlp_out, training=training)
        mlp_out = self.mlp2(mlp_out)
        mlp_out = self.drop3(mlp_out, training=training)
        x = x + mlp_out
        return x

class PatchAndPositionEmbedding(layers.Layer):
    def __init__(self, patch_size=16, embed_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.proj = layers.Conv2D(filters=embed_dim, kernel_size=(1, patch_size),
                                  strides=(1, patch_size), padding="valid")
        self.pos_embedding = None
        self.n_tokens = None

    def build(self, input_shape):
        proj_shape = self.proj.compute_output_shape(input_shape)
        self.h_tokens = int(proj_shape[1])
        self.w_tokens = int(proj_shape[2])
        self.n_tokens = self.h_tokens * self.w_tokens
        self.pos_embedding = self.add_weight(
            name="pos_embedding",
            shape=(1, self.n_tokens, self.embed_dim),
            initializer="random_normal",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        x = self.proj(x)
        x = tf.reshape(x, (-1, self.n_tokens, self.embed_dim))
        return x + self.pos_embedding

def build_frequency_aware_representation(x, freq_filters=32, multi_scale_kernels=(7, 15, 31),
                                         dropout_rate=0.10, name_prefix="freq"):
    branches = []
    for i, k in enumerate(multi_scale_kernels):
        b = conv_bn_act(x, filters=freq_filters, kernel_size=(1, k),
                        dropout_rate=dropout_rate, name_prefix=f"{name_prefix}_ms{i+1}")
        branches.append(b)

    x_ms = layers.Concatenate(name=f"{name_prefix}_concat")(branches)
    energy_map = layers.Lambda(lambda t: tf.reduce_mean(tf.square(t), axis=2, keepdims=True),
                               name=f"{name_prefix}_energy")(x_ms)
    width = x_ms.shape[2]
    energy_map = layers.Lambda(lambda t: tf.repeat(t, repeats=width, axis=2),
                               name=f"{name_prefix}_energy_tile")(energy_map)
    x_fused = layers.Concatenate(name=f"{name_prefix}_energy_concat")([x_ms, energy_map])

    x_proj = conv_bn_act(x_fused, filters=freq_filters * 2, kernel_size=(1, 1),
                         dropout_rate=dropout_rate, name_prefix=f"{name_prefix}_proj")

    gate = layers.GlobalAveragePooling2D(name=f"{name_prefix}_gap")(x_proj)
    gate = layers.Dense(freq_filters * 2, activation="sigmoid", name=f"{name_prefix}_gate")(gate)
    gate = layers.Reshape((1, 1, freq_filters * 2), name=f"{name_prefix}_gate_reshape")(gate)
    x_out = layers.Multiply(name=f"{name_prefix}_gated")([x_proj, gate])
    return x_out

def build_cnn_local_encoder(x, filters_1=32, filters_2=64, kernel_1=(3,7), kernel_2=(3,5),
                            dropout_rate=0.20, embedding_dim=128, name_prefix="cnn_local"):
    x = conv_bn_act(x, filters=filters_1, kernel_size=kernel_1,
                    dropout_rate=dropout_rate, name_prefix=f"{name_prefix}_block1")
    x = layers.MaxPooling2D(pool_size=(1,2), name=f"{name_prefix}_pool1")(x)
    x = conv_bn_act(x, filters=filters_2, kernel_size=kernel_2,
                    dropout_rate=dropout_rate, name_prefix=f"{name_prefix}_block2")
    x = layers.MaxPooling2D(pool_size=(1,2), name=f"{name_prefix}_pool2")(x)
    x = layers.GlobalAveragePooling2D(name=f"{name_prefix}_gap")(x)
    x = layers.Dense(embedding_dim, activation="relu", name=f"{name_prefix}_embedding")(x)
    return x

def build_vit_global_encoder(x, patch_size=16, embed_dim=128, num_heads=4,
                             transformer_depth=2, mlp_dim=256, dropout_rate=0.10,
                             name_prefix="vit_global"):
    x = PatchAndPositionEmbedding(patch_size=patch_size, embed_dim=embed_dim,
                                  name=f"{name_prefix}_patch_embed")(x)
    for i in range(transformer_depth):
        x = TransformerEncoder(embed_dim=embed_dim, num_heads=num_heads, mlp_dim=mlp_dim,
                               dropout_rate=dropout_rate, name=f"{name_prefix}_block{i+1}")(x)
    x = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_norm")(x)
    x = layers.GlobalAveragePooling1D(name=f"{name_prefix}_gap")(x)
    x = layers.Dense(embed_dim, activation="relu", name=f"{name_prefix}_embedding")(x)
    return x

def build_adaptive_fusion(local_feat, global_feat, fusion_dim=128, dropout_rate=0.25, name_prefix="fusion"):
    local_proj = layers.Dense(fusion_dim, activation="relu", name=f"{name_prefix}_local_proj")(local_feat)
    global_proj = layers.Dense(fusion_dim, activation="relu", name=f"{name_prefix}_global_proj")(global_feat)

    fusion_context = layers.Concatenate(name=f"{name_prefix}_concat")([local_proj, global_proj])
    gate = layers.Dense(fusion_dim, activation="sigmoid", name=f"{name_prefix}_gate")(fusion_context)

    gated_local = layers.Multiply(name=f"{name_prefix}_gated_local")([gate, local_proj])
    gated_global = layers.Multiply(name=f"{name_prefix}_gated_global")(
        [layers.Lambda(lambda t: 1.0 - t, name=f"{name_prefix}_inv_gate")(gate), global_proj]
    )

    fused = layers.Add(name=f"{name_prefix}_add")([gated_local, gated_global])
    fused = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop")(fused)
    fused = layers.Dense(fusion_dim, activation="relu", name=f"{name_prefix}_embedding")(fused)
    return fused

def build_uncertainty_classifier(fused_embedding, hidden_dim=128, dropout_rate=0.30, name_prefix="uncertainty_head"):
    x = layers.Dense(hidden_dim, activation="relu", name=f"{name_prefix}_fc1")(fused_embedding)
    x = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop1")(x)
    x = layers.Dense(hidden_dim // 2, activation="relu", name=f"{name_prefix}_fc2")(x)
    x = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop2")(x)
    prob = layers.Dense(1, activation="sigmoid", name=f"{name_prefix}_prob")(x)
    return prob

def build_fa_cnnvit(n_channels, window_samples, config, model_name="FA-CNNViT"):
    inputs = layers.Input(shape=(n_channels, window_samples, 1), name="eeg_input")

    rep = build_frequency_aware_representation(
        inputs,
        freq_filters=config["freq_filters"],
        multi_scale_kernels=config["multi_scale_kernels"],
        dropout_rate=config["representation_dropout"],
        name_prefix="freq_repr"
    )

    local_feat = build_cnn_local_encoder(
        rep,
        filters_1=config["cnn_filters_1"],
        filters_2=config["cnn_filters_2"],
        kernel_1=config["cnn_kernel_1"],
        kernel_2=config["cnn_kernel_2"],
        dropout_rate=config["cnn_dropout"],
        embedding_dim=config["fusion_dim"],
        name_prefix="cnn_local"
    )

    global_feat = build_vit_global_encoder(
        rep,
        patch_size=config["patch_size"],
        embed_dim=config["embed_dim"],
        num_heads=config["num_heads"],
        transformer_depth=config["transformer_depth"],
        mlp_dim=config["mlp_dim"],
        dropout_rate=config["transformer_dropout"],
        name_prefix="vit_global"
    )

    fused = build_adaptive_fusion(
        local_feat,
        global_feat,
        fusion_dim=config["fusion_dim"],
        dropout_rate=config["fusion_dropout"],
        name_prefix="adaptive_fusion"
    )

    outputs = build_uncertainty_classifier(
        fused,
        hidden_dim=config["classifier_hidden"],
        dropout_rate=config["classifier_dropout"],
        name_prefix="uncertainty_head"
    )

    return keras.Model(inputs=inputs, outputs=outputs, name=model_name)

print("FA-CNNViT blocks ready.")


In [ ]:
# ==============================================
# Cell 16: Loss, optimizer, callbacks
# ==============================================

def asymmetric_focal_loss(gamma_pos=1.0, gamma_neg=2.0, alpha=0.25, eps=1e-7):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)

        pos_loss = -alpha * y_true * tf.pow(1.0 - y_pred, gamma_pos) * tf.math.log(y_pred)
        neg_loss = -(1.0 - alpha) * (1.0 - y_true) * tf.pow(y_pred, gamma_neg) * tf.math.log(1.0 - y_pred)
        return tf.reduce_mean(pos_loss + neg_loss)
    return loss_fn

loss_fn = asymmetric_focal_loss(gamma_pos=1.0, gamma_neg=2.0, alpha=0.25)

optimizer = keras.optimizers.Adam(learning_rate=TRAIN_CONFIG["learning_rate"])

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=TRAIN_CONFIG["early_stopping_patience"],
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=TRAIN_CONFIG["reduce_lr_patience"],
        min_lr=1e-6
    )
]

print("Loss, optimizer, and callbacks configured.")


In [ ]:
# ==============================================
# Cell 17: Build and compile model
# ==============================================

X_train = fold_arrays["X_train"]
if X_train.ndim != 4:
    raise ValueError(f"Expected X_train to have 4 dimensions, got {X_train.shape}")

n_channels = X_train.shape[1]
window_samples = X_train.shape[2]

print(f"Input shape: channels={n_channels}, window_samples={window_samples}")

model_name = f"FA-CNNViT_{dataset_name.replace('-', '_')}_fold{outer_fold}"
model = build_fa_cnnvit(
    n_channels=n_channels,
    window_samples=window_samples,
    config=MODEL_CONFIG,
    model_name=model_name
)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc_roc", curve="ROC"),
        keras.metrics.AUC(name="auc_pr", curve="PR"),
    ]
)

print("Model compiled successfully.")
print("Total params:", model.count_params())


In [ ]:
# ==============================================
# Cell 19: Train one fold
# ==============================================

USE_CLASS_WEIGHTS = False
cw = class_weight_dict if USE_CLASS_WEIGHTS else None

history = model.fit(
    fold_arrays["X_train"],
    fold_arrays["y_train"],
    validation_data=(fold_arrays["X_val"], fold_arrays["y_val"]),
    epochs=TRAIN_CONFIG["epochs"],
    batch_size=TRAIN_CONFIG["batch_size"],
    callbacks=callbacks,
    class_weight=cw,
    verbose=1
)

history_df = pd.DataFrame(history.history)
history_path = LOG_DIR / f"history_{dataset_name.lower().replace('-', '_')}_fold{outer_fold}.csv"
history_df.to_csv(history_path, index=False)

print("Saved training history:", history_path)
history_df.head()


In [ ]:
# ==============================================
# Cell 20: Plot training curves
# ==============================================

history_dict = history.history
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].plot(history_dict["loss"], label="Train")
axes[0].plot(history_dict["val_loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.25)

metric_name = "auc_pr" if "auc_pr" in history_dict else "accuracy"
val_metric_name = f"val_{metric_name}"
axes[1].plot(history_dict[metric_name], label=f"Train {metric_name}")
axes[1].plot(history_dict[val_metric_name], label=f"Validation {metric_name}")
axes[1].set_title(metric_name.upper())
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel(metric_name.upper())
axes[1].legend()
axes[1].grid(axis="y", linestyle="--", alpha=0.25)

plt.tight_layout()

curve_png = FIG_DIR / f"training_curves_{dataset_name.lower().replace('-', '_')}_fold{outer_fold}.png"
curve_pdf = FIG_DIR / f"training_curves_{dataset_name.lower().replace('-', '_')}_fold{outer_fold}.pdf"
fig.savefig(curve_png, dpi=600, bbox_inches="tight")
fig.savefig(curve_pdf, bbox_inches="tight")

print("Saved:")
print(" -", curve_png)
print(" -", curve_pdf)

plt.show()


In [ ]:
# ==============================================
# Cell 21: Validation-only threshold selection
# ==============================================

def false_alarms_per_hour(y_true, y_pred, window_sec):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    fp = np.sum((y_true == 0) & (y_pred == 1))
    total_hours = (len(y_true) * window_sec) / 3600.0
    if total_hours <= 0:
        return 0.0
    return fp / total_hours

def threshold_search(y_true, y_prob, cfg):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(np.float32)

    rows = []
    for thr in cfg["threshold_grid"]:
        y_hat = (y_prob >= thr).astype(int)

        recall = recall_score(y_true, y_hat, zero_division=0)
        precision = precision_score(y_true, y_hat, zero_division=0)
        f1 = f1_score(y_true, y_hat, zero_division=0)
        fa_h = false_alarms_per_hour(y_true, y_hat, cfg["window_sec"])
        objective = recall - cfg["fa_penalty"] * fa_h

        rows.append({
            "threshold": float(thr),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "false_alarms_per_hour": fa_h,
            "objective": objective
        })

    df = pd.DataFrame(rows).sort_values("objective", ascending=False).reset_index(drop=True)
    best_thr = float(df.iloc[0]["threshold"])
    return best_thr, df

val_prob = model.predict(fold_arrays["X_val"], batch_size=TRAIN_CONFIG["batch_size"]).ravel()
best_threshold, threshold_df = threshold_search(fold_arrays["y_val"].astype(int), val_prob, TRAIN_CONFIG)

threshold_path = TABLE_DIR / f"threshold_search_{dataset_name.lower().replace('-', '_')}_fold{outer_fold}.csv"
threshold_df.to_csv(threshold_path, index=False)

print("Best validation threshold:", best_threshold)
print("Saved threshold table:", threshold_path)
threshold_df.head()


In [ ]:
# ==============================================
# Cell 22: Monte Carlo dropout inference
# ==============================================

def mc_dropout_predict(model, X, n_passes=20, batch_size=32):
    if len(X) == 0:
        raise ValueError("Input X is empty for MC Dropout.")

    X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
    n_samples = X.shape[0]
    preds = []

    for _ in range(n_passes):
        batch_preds = []
        for start in range(0, n_samples, batch_size):
            end = start + batch_size
            x_batch = X_tf[start:end]
            p = model(x_batch, training=True)
            p = tf.reshape(p, [-1]).numpy()
            batch_preds.append(p)
        preds.append(np.concatenate(batch_preds))

    preds = np.stack(preds, axis=0)
    pred_mean = preds.mean(axis=0)
    pred_var = preds.var(axis=0)
    return pred_mean, pred_var, preds

test_prob_mean, test_prob_var, test_prob_all = mc_dropout_predict(
    model,
    fold_arrays["X_test"],
    n_passes=TRAIN_CONFIG["mc_dropout_passes"],
    batch_size=TRAIN_CONFIG["batch_size"]
)

print("MC mean shape:", test_prob_mean.shape)
print("MC variance shape:", test_prob_var.shape)


In [ ]:
# ==============================================
# Cell 23: Held-out test metrics
# ==============================================

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(np.float32)

    if len(y_true) == 0:
        return 0.0

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            idx = (y_prob >= lo) & (y_prob <= hi)
        else:
            idx = (y_prob >= lo) & (y_prob < hi)

        if np.sum(idx) == 0:
            continue

        acc = np.mean(y_true[idx])
        conf = np.mean(y_prob[idx])
        ece += (np.sum(idx) / len(y_true)) * abs(acc - conf)

    return float(ece)

def compute_binary_metrics(y_true, y_prob, threshold, window_sec):
    y_true = np.asarray(y_true).astype(int)
    y_hat = (np.asarray(y_prob) >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()

    precision = precision_score(y_true, y_hat, zero_division=0)
    recall = recall_score(y_true, y_hat, zero_division=0)
    f1 = f1_score(y_true, y_hat, zero_division=0)
    specificity = tn / (tn + fp + 1e-8)
    fa_h = false_alarms_per_hour(y_true, y_hat, window_sec)

    roc_auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    pr_auc = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    brier = brier_score_loss(y_true, y_prob)
    ece = expected_calibration_error(y_true, y_prob, n_bins=10)

    return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "specificity": specificity,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "false_alarms_per_hour": fa_h,
        "brier_score": brier,
        "ece": ece,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }

test_metrics = compute_binary_metrics(
    y_true=fold_arrays["y_test"],
    y_prob=test_prob_mean,
    threshold=best_threshold,
    window_sec=TRAIN_CONFIG["window_sec"]
)

test_metrics_df = pd.DataFrame([test_metrics])
test_metrics_path = TABLE_DIR / f"test_metrics_{dataset_name.lower().replace('-', '_')}_fold{outer_fold}.csv"
test_metrics_df.to_csv(test_metrics_path, index=False)

print("Saved test metrics:", test_metrics_path)
test_metrics_df


In [ ]:
# ==============================================
# Cell 24: Save trained fold model and parameter summary
# ==============================================

model_filename = f"fa_cnnvit_{dataset_name.lower().replace('-', '_')}_fold{outer_fold}.keras"
model_path = MODEL_DIR / model_filename
model.save(model_path)

param_summary_df = pd.DataFrame([{
    "dataset": dataset_name,
    "outer_fold": outer_fold,
    "n_channels": int(fold_arrays["X_train"].shape[1]),
    "window_samples": int(fold_arrays["X_train"].shape[2]),
    "total_params": int(model.count_params())
}])

param_path = TABLE_DIR / f"fa_cnnvit_parameter_summary_{dataset_name.lower().replace('-', '_')}_fold{outer_fold}.csv"
param_summary_df.to_csv(param_path, index=False)

config_path = MODEL_DIR / f"{model_filename.replace('.keras', '_config.json')}"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump({
        "dataset_name": dataset_name,
        "outer_fold": int(outer_fold),
        "train_config": TRAIN_CONFIG,
        "model_config": MODEL_CONFIG,
        "best_threshold": float(best_threshold),
        "excluded_chbmit_files": len(EXCLUDED_CHBMIT_FILES),
    }, f, indent=2)

print("Saved trained model:", model_path)
print("Saved parameter summary:", param_path)
print("Saved config JSON:", config_path)


In [ ]:
# ==============================================
# Cell 25: Optional all-fold loop (disabled by default)
# ==============================================

RUN_ALL_FOLDS = False

if RUN_ALL_FOLDS:
    all_results = []

    available_folds = sorted(split_manifest_df[split_manifest_df["dataset"] == dataset_name]["outer_fold"].unique())

    for fold_id in available_folds:
        print(f"\n===== Running dataset={dataset_name}, fold={fold_id} =====")

        TRAIN_CONFIG["outer_fold"] = int(fold_id)
        fold_split_df = split_manifest_df[
            (split_manifest_df["dataset"] == dataset_name) &
            (split_manifest_df["outer_fold"] == fold_id)
        ].copy()

        EXCLUDED_CHBMIT_FILES.clear()

        norm_mean, norm_std = fit_normalization_stats_for_fold(dataset_name, fold_split_df, TRAIN_CONFIG)
        fold_arrays = build_arrays_for_fold(dataset_name, fold_split_df, TRAIN_CONFIG, norm_mean, norm_std)

        model = build_fa_cnnvit(
            n_channels=fold_arrays["X_train"].shape[1],
            window_samples=fold_arrays["X_train"].shape[2],
            config=MODEL_CONFIG,
            model_name=f"FA-CNNViT_{dataset_name.replace('-', '_')}_fold{fold_id}"
        )

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=TRAIN_CONFIG["learning_rate"]),
            loss=loss_fn,
            metrics=[
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.Precision(name="precision"),
                keras.metrics.Recall(name="recall"),
                keras.metrics.AUC(name="auc_roc", curve="ROC"),
                keras.metrics.AUC(name="auc_pr", curve="PR"),
            ]
        )

        history = model.fit(
            fold_arrays["X_train"],
            fold_arrays["y_train"],
            validation_data=(fold_arrays["X_val"], fold_arrays["y_val"]),
            epochs=TRAIN_CONFIG["epochs"],
            batch_size=TRAIN_CONFIG["batch_size"],
            callbacks=callbacks,
            verbose=1
        )

        val_prob = model.predict(fold_arrays["X_val"], batch_size=TRAIN_CONFIG["batch_size"]).ravel()
        best_threshold, _ = threshold_search(fold_arrays["y_val"].astype(int), val_prob, TRAIN_CONFIG)

        test_prob_mean, test_prob_var, _ = mc_dropout_predict(
            model,
            fold_arrays["X_test"],
            n_passes=TRAIN_CONFIG["mc_dropout_passes"],
            batch_size=TRAIN_CONFIG["batch_size"]
        )

        test_metrics = compute_binary_metrics(
            y_true=fold_arrays["y_test"],
            y_prob=test_prob_mean,
            threshold=best_threshold,
            window_sec=TRAIN_CONFIG["window_sec"]
        )
        test_metrics["dataset"] = dataset_name
        test_metrics["outer_fold"] = int(fold_id)
        test_metrics["excluded_files"] = len(EXCLUDED_CHBMIT_FILES)
        all_results.append(test_metrics)

    all_results_df = pd.DataFrame(all_results)
    all_results_path = TABLE_DIR / f"all_fold_results_{dataset_name.lower().replace('-', '_')}.csv"
    all_results_df.to_csv(all_results_path, index=False)
    print("\nSaved all-fold results:", all_results_path)
    display(all_results_df)


In [ ]:
# ==============================================
# Cell 18: Inspect class balance
# ==============================================

y_train = fold_arrays["y_train"].astype(int)

if y_train.ndim != 1:
    raise ValueError(f"Expected y_train to be 1D, got shape {y_train.shape}")
if len(y_train) == 0:
    raise ValueError("y_train is empty. Cannot compute class weights.")

classes = np.unique(y_train)
if len(classes) < 2:
    print("Warning: Only one class present in training data.")
    class_weight_dict = {int(classes[0]): 1.0}
else:
    class_weights_arr = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}

print("Class weights:", class_weight_dict)
print("Training positive ratio:", y_train.mean())
